[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefHosni/prompt-engineering-book-2e/blob/main/notebooks/ch16_end_to_end.ipynb)

# Chapter 16 — Building an End-to-End System

The whole book in one pipeline. Nothing new is introduced here; every
technique has appeared earlier.

| stage | from |
|---|---|
| triage | §8.2 schema-constrained extraction |
| reply | §6.4 grounded, with an escape hatch |
| validate | §13.4 code, because code can express these rules |
| retry | §12.3 bounded, because an unbounded retry is an outage |

In [1]:
# Colab only: fetch the shared helpers and install the SDKs.
# Running locally? Skip this cell -- you already have them.
import os, sys
if "google.colab" in sys.modules:
    !pip -q install anthropic openai python-dotenv
    !git clone -q https://github.com/youssefHosni/prompt-engineering-book-2e /content/peb-book
    sys.path.insert(0, "/content/peb-book")
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")


In [2]:
import json
import re

from peb import ask
from peb.checks import validate
from peb.fixtures import ORDER, POLICY

MAX_ATTEMPTS = 3

CUSTOMER = ("My Nordic 400 arrived 5 days late on order KL-449021, and the drip "
            "tray has already warped. What are you going to do about it?")

## Triage

`needs_human` is a deliberately narrow rule. A vague escalation condition
escalates everything, and a pipeline that escalates everything is a queue.

In [3]:
TRIAGE_SCHEMA = {
    "type": "object",
    "properties": {
        "intent":      {"type": "string",
                        "enum": ["delivery", "claims", "packaging",
                                 "surcharge", "other"]},
        "order_ref":   {"type": ["string", "null"]},
        "sentiment":   {"type": "string",
                        "enum": ["positive", "negative", "mixed"]},
        "needs_human": {"type": "boolean"},
    },
    "required": ["intent", "order_ref", "sentiment", "needs_human"],
    "additionalProperties": False,
}

TRIAGE_RULE = ("needs_human is true only if the customer threatens legal action or asks "
               "for a refund over 500 pounds.")


def triage(message):
    prompt = (f"Triage this customer message.\n"
              f"Reply with one JSON object matching this schema and nothing else.\n\n"
              f"{json.dumps(TRIAGE_SCHEMA, indent=2)}\n\n{TRIAGE_RULE}\n\n"
              f"Message: ```{message}```")
    text, _ = ask(prompt, max_tokens=4000)
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        raise ValueError(f"no JSON in triage reply: {text[:80]!r}")
    obj = json.loads(m.group(0))
    missing = set(TRIAGE_SCHEMA["required"]) - set(obj)
    if missing:
        raise ValueError(f"triage missing keys: {sorted(missing)}")
    return obj

## Draft

The feedback line matters. On a retry the model is told *what failed*, not
"try again" — §12.3.

In [4]:
REPLY_RULES = (
    "Use ONLY the policy and order record above for any factual claim.\n"
    "If the answer is not in them, say you will check and come back.\n"
    "Length: 50 to 70 words.\n"
    "Do not offer a refund. Do not invent dates or amounts.\n"
    "Sign as 'Kestrel Support'.")


def draft(message, order, feedback=None):
    prompt = (f"Policy:\n{POLICY}\n\nOrder record:\n{json.dumps(order, indent=2)}\n\n"
              f"{REPLY_RULES}\n\nCustomer: ```{message}```")
    if feedback:
        prompt += f"\n\nYour previous draft failed these checks: {'; '.join(feedback)}"
    text, _ = ask(prompt, max_tokens=4000)
    return text.strip()

## Run it

`validate()` lives in `peb/checks.py` and is covered by `test_offline.py`,
which runs without a key. The checker is the part most worth testing.

In [5]:
print("CUSTOMER  " + CUSTOMER + "\n")

t = triage(CUSTOMER)
print("triage    " + json.dumps(t))

if t["needs_human"]:
    print("\n-> escalated. The pipeline stops here, by design.")
else:
    feedback = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        reply = draft(CUSTOMER, ORDER, feedback)
        problems = validate(reply, ORDER)
        print(f"\nattempt {attempt}  ({len(reply.split())} words)")
        print(reply)
        if not problems:
            print("\n-> passed every check. Sent.")
            break
        print(f"failed: {problems}")
        feedback = problems
    else:
        # Bounded. The failure is visible, not silent.
        print(f"\n-> {MAX_ATTEMPTS} attempts, still failing {feedback}. Handed to a human.")

CUSTOMER  My Nordic 400 arrived 5 days late on order KL-449021, and the drip tray has already warped. What are you going to do about it?



triage    {"intent": "claims", "order_ref": "KL-449021", "sentiment": "negative", "needs_human": false}



attempt 1  (61 words)
Thanks for flagging this. Your order was promised within 3 working days and arrived on 2026-03-11, exceeding the promise by more than 2 working days, so the delivery fee refund is applied automatically within 5 working days. The drip tray is a covered component and your order is replacement-eligible, so a free replacement part ships within 2 working days.

Kestrel Support
failed: ['mentions refund']



attempt 2  (64 words)
Thank you for flagging this, and I'm sorry the Nordic 400 on order KL-449021 arrived later than promised.

The drip tray is a covered component and your order is marked replacement-eligible, so a free-of-charge replacement will be shipped within 2 working days.

On the late delivery itself, I'll check what your order qualifies for under our policy and come back to you.

Kestrel Support

-> passed every check. Sent.
